In [1]:
# Load data as a long-format pandas data frame
import pandas as pd

context_df = pd.read_csv("../data/m4_hourly_train.csv")
context_df

,item_id,timestamp,target
0,H1,1750-01-01 00:00:00,605.0
1,H1,1750-01-01 01:00:00,586.0
2,H1,1750-01-01 02:00:00,586.0
3,H1,1750-01-01 03:00:00,559.0
4,H1,1750-01-01 04:00:00,511.0
...,...,...,...
353495,H414,1750-02-09 19:00:00,48.0
353496,H414,1750-02-09 20:00:00,41.0
353497,H414,1750-02-09 21:00:00,35.0
353498,H414,1750-02-09 22:00:00,26.0


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages

def plot_multiple_items_random(context_df, pdf_path,
                               n_items=20,
                               timestamp_col="timestamp",
                               target_col="target",
                               recent_n=200):

    # ---- 1. Randomly select 20 item_ids ----
    unique_ids = context_df["item_id"].unique()
    selected_ids = np.random.choice(unique_ids, size=n_items, replace=False)

    print("Selected item_ids:", selected_ids)

    fig_size = (12, 4)

    with PdfPages(pdf_path) as pdf:

        for item_id in selected_ids:
            df = context_df[context_df["item_id"] == item_id].copy()
            df[timestamp_col] = pd.to_datetime(df[timestamp_col])

            # ---- 200 most recent ----
            df_recent = df.tail(recent_n).reset_index(drop=True)
            x = df_recent[timestamp_col]
            y = df_recent[target_col]

            # 90% cutoff
            n = len(df_recent)
            cutoff_idx = int(n * 0.9)
            x_cut = x.iloc[cutoff_idx - 1]   # cutoff timestamp

            # =====================================================
            #                A. FULL 200 + vertical bar
            # =====================================================
            figA, axA = plt.subplots(figsize=fig_size)
            axA.plot(x, y, linewidth=1.2)

            # Dense grid
            axA.grid(which="major", linestyle="--", alpha=0.6)
            axA.grid(which="minor", linestyle=":", alpha=0.4)
            axA.minorticks_on()

            # ---- Vertical bar at 90% point ----
            axA.axvline(x_cut, color="red", linestyle="--", linewidth=1)

            axA.set_title(f"[{item_id}] Recent 200 Points (with 90% marker)")
            axA.set_xlabel("Timestamp")
            axA.set_ylabel("Target")
            figA.tight_layout()

            # Save axis limits for B
            xlim_A = axA.get_xlim()
            ylim_A = axA.get_ylim()

            pdf.savefig(figA)
            plt.close(figA)

            # =====================================================
            #              B. FIRST 90% + vertical bar
            # =====================================================
            figB, axB = plt.subplots(figsize=fig_size)
            axB.plot(x[:cutoff_idx], y[:cutoff_idx], linewidth=1.2)

            # Use A's limits → blank right region
            axB.set_xlim(xlim_A)
            axB.set_ylim(ylim_A)

            # Dense grid
            axB.grid(which="major", linestyle="--", alpha=0.6)
            axB.grid(which="minor", linestyle=":", alpha=0.4)
            axB.minorticks_on()

            # ---- Vertical bar at END of B (same location) ----
            axB.axvline(x_cut, color="red", linestyle="--", linewidth=1)

            axB.set_title(f"[{item_id}] Left 90% (with boundary marker)")
            axB.set_xlabel("Timestamp")
            axB.set_ylabel("Target")
            figB.tight_layout()

            pdf.savefig(figB)
            plt.close(figB)

    print(f"Saved combined PDF: {pdf_path}")


/home/fli/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
